# Tutorial: LLM Client

Audience:
- Developers enabling LLM classification in intelligence runs.

Prerequisites:
- `OPENAI_API_KEY` if using the OpenAI provider.

Learning goals:
- Configure `LLMConfig`.
- Fall back to mock provider when keys are missing.
- Inspect LLM-enriched event metadata.


## Outline

1. Setup imports
2. Build LLM-enabled intelligence config
3. Fallback to mock provider if needed
4. Run pipeline and inspect classified events


In [ ]:
import os

from swing_screener.intelligence import IntelligenceConfig, LLMConfig, run_intelligence_pipeline

## Step 1 - Configure LLM settings

Set provider/model/cache/audit options on the intelligence config.


In [ ]:
cfg = IntelligenceConfig(
    enabled=True,
    providers=("yahoo_finance",),
    llm=LLMConfig(
        enabled=True,
        provider="openai",
        model="gpt-4o-mini",
        api_key=os.environ.get("OPENAI_API_KEY", ""),
        enable_cache=True,
        enable_audit=True,
        cache_path="data/intelligence/llm_cache.json",
        audit_path="data/intelligence/llm_audit",
    ),
)

print(
    f"LLM Config: enabled={cfg.llm.enabled}, provider={cfg.llm.provider}, model={cfg.llm.model}"
)


## Step 2 - Apply fallback behavior

If no key is present and provider is not mock, switch to mock for local demonstration.


In [ ]:
if not cfg.llm.api_key and cfg.llm.provider != "mock":
    print("Warning: No API key set. Using mock provider for demonstration.")
    cfg = IntelligenceConfig(
        enabled=True,
        providers=("yahoo_finance",),
        llm=LLMConfig(
            enabled=True,
            provider="mock",
            model="test-model",
        ),
    )

cfg.llm.provider


## Step 3 - Run and inspect LLM classifications

Scan events and print the first enriched record.


In [ ]:
symbols = ["NVDA", "AMD", "AAPL", "MSFT"]
snapshot = run_intelligence_pipeline(symbols=symbols, cfg=cfg)

print(f"Processed {len(snapshot.events)} events")

for event in snapshot.events:
    if event.metadata.get("llm_event_type"):
        print(f"Event: {event.headline[:60]}...")
        print(f"  LLM type: {event.metadata.get('llm_event_type')}")
        print(f"  LLM severity: {event.metadata.get('llm_severity')}")
        print(f"  LLM confidence: {event.metadata.get('llm_confidence'):.2f}")
        print(f"  LLM is_material: {event.metadata.get('llm_is_material')}")
        print(f"  Credibility (blended): {event.credibility:.3f}")
        break